# Process Strategy Cost Explorer
### Which process strategy is cheapest at which volume?

This notebook walks you through the **crossover model** for process strategies (Heizer, Render & Munson, *Operations Management*, Chapter 7). You will build the picture in four steps, from simple to complex:

| Step | What you see | New idea |
|---|---|---|
| 1 | A small cost table | Total cost = fixed cost + variable cost × volume |
| 2 | Two cost lines: **process focus** vs. **product focus** | The crossover volume |
| 2b | Aylin's bakery from the case | The same model with real-sized numbers |
| 3 | The same two lines, now interactive | How each cost type moves the crossover |
| 4 | Add **repetitive focus** | Each strategy "owns" a volume band |
| 5 | Add **mass customization** | What the cost model cannot show |

**How to use this notebook.** A notebook is a document that mixes text (like this cell) with small programs ("code cells", grey boxes). You run a code cell by clicking into it and pressing **Shift + Enter**, or by clicking the ▶ button on its left. Run the cells **from top to bottom**, because later cells use what earlier cells created. If something goes wrong, choose *Runtime → Restart and run all* from the menu.


## Step 0 – Load the toolboxes

Python on its own knows only basic arithmetic. For charts and buttons we load **libraries**: ready-made toolboxes that other people wrote, much like borrowing a toolbox instead of forging your own hammer. We use three:

- `numpy` – calculates with whole lists of numbers at once (e.g. the cost for 0, 10, 20 … 40,000 units in one go)
- `matplotlib` – draws charts
- `ipywidgets` – adds input boxes and buttons to the notebook

Run the cell below once. Nothing visible happens except a short confirmation.


In [ ]:
# =============================================================================
# STEP 0 · Load the toolboxes
# =============================================================================
# "import" opens a toolbox so that we can use the tools inside it.
# The short name after "as" is a nickname: we write np.something instead of numpy.something.

import numpy as np                     # numpy: calculates with whole lists of numbers at once
import matplotlib.pyplot as plt        # matplotlib: draws charts (we call it "plt")
import ipywidgets as widgets           # ipywidgets: input boxes and buttons inside the notebook
from IPython.display import display, clear_output   # display = show a widget; clear_output = wipe an old chart

# print(...) writes a line of text under the cell, so you know the cell has run.
print("Toolboxes loaded. Continue with Step 1.")


## Step 1 – One formula for every strategy

Every process strategy causes two kinds of cost:

- **Fixed cost (F)** – paid regardless of how many units you make: buildings, machines, specialised equipment, IT systems. Measured in € per year.
- **Variable cost (v)** – paid for every single unit: labour time, material, energy. Measured in € per unit.

The **total cost** for a yearly volume **Q** is therefore

$$\text{Total cost} = F + v \times Q$$

The two extreme strategies differ exactly in this mix:

- **Process focus** (e.g. a hospital, a job shop) uses flexible, general-purpose equipment: **low fixed cost**, but each job needs much skilled work: **high variable cost**.
- **Product focus** (e.g. Frito-Lay's chip lines, a refinery) invests in a dedicated, specialised line: **high fixed cost**, but each unit is cheap: **low variable cost**.

👉 **Your turn:** change the four numbers in the cell below and run it again. The numbers are illustrative, not real company data.


**What the next cell does, in plain words.**

1. It stores the four cost figures in named *variables*.
2. It defines the cost formula once as a *function* `total_cost(F, v, Q)`.
3. It runs the formula for seven volumes and prints a small table with the cheaper strategy in the last column.

**What you should see:** a table of seven rows; somewhere between 7,500 and 10,000 units the last column switches from *process focus* to *product focus*.

In [ ]:
# =============================================================================
# STEP 1 · One formula for every strategy:  Total cost = F + v × Q
# =============================================================================

# ---- 1. Enter the cost figures here ----------------------------------------
# A "variable" is a named box that holds a number. The name is on the left of "=",
# the number on the right. Change a number, then run the cell again (Shift + Enter).
# The underscore in 50_000 is only a visual thousands separator; Python ignores it.

F_process = 50_000     # fixed cost of process focus    [€ per year]   (flexible machines, little investment)
v_process = 60         # variable cost of process focus [€ per unit]   (much skilled work per job)

F_product = 500_000    # fixed cost of product focus    [€ per year]   (a dedicated line: big investment)
v_product = 10         # variable cost of product focus [€ per unit]   (each unit is cheap on the line)


# ---- 2. The cost formula as a reusable "function" ---------------------------
# A function is a small machine: you put numbers in, it gives a result back.
# "def" starts the definition; F, v and Q are the inputs ("parameters");
# "return" hands back the result. Writing the formula once means we can use it
# for every strategy and every volume without typing it again.
def total_cost(F, v, Q):
    return F + v * Q            # fixed cost + (variable cost per unit × number of units)


# ---- 3. Print a small table for a few volumes -------------------------------
# The f"..." strings are "formatted text": anything in {curly brackets} is replaced
# by its value. ":>8" means "right-aligned, 8 characters wide"; ":," adds thousands separators.
print(f"{'Volume':>8} | {'Process focus':>14} | {'Product focus':>14} | Cheaper")
print("-" * 58)                 # a line of 58 dashes under the header

# A "for" loop repeats the indented lines once for every number in the list:
# first with Q = 0, then Q = 2,500, and so on.
for Q in [0, 2_500, 5_000, 7_500, 10_000, 12_500, 15_000]:
    cost_process = total_cost(F_process, v_process, Q)   # use the machine for process focus
    cost_product = total_cost(F_product, v_product, Q)   # ... and for product focus

    # "if ... else" chooses between two cases: is process focus cheaper or not?
    if cost_process < cost_product:
        cheaper = "process focus"
    else:
        cheaper = "product focus"

    # One row of the table: volume, both costs, and the winner
    print(f"{Q:>8,} | {cost_process:>12,} € | {cost_product:>12,} € | {cheaper}")


**Look at the last column.** Somewhere between two rows of the table, the cheaper strategy switches. Step 2 finds the exact volume.


## Step 2 – Draw the two lines and find the crossover

Each strategy's total cost is a straight line: it **starts at F** (the cost at zero volume) and **rises with slope v**. The volume where both lines meet is the **crossover volume Q\***. There both strategies cost the same:

$$F_{process} + v_{process} \cdot Q^* = F_{product} + v_{product} \cdot Q^*$$

Solving for Q\* gives

$$Q^* = \frac{F_{product} - F_{process}}{v_{process} - v_{product}}$$

In words: **extra fixed cost of product focus ÷ variable cost saved per unit**. Below Q\* the savings per unit do not yet pay back the big investment; above Q\* they do.


**What the next cell does, in plain words.**

1. It computes Q\* with the formula above and prints it.
2. It creates 201 volumes between 0 and 20,000 for the x-axis.
3. It calculates both cost lines and draws them, with a dot and a dotted line at Q\*.

**What you should see:** *Crossover volume Q\* = 9,000 units per year* and a chart where the blue line starts lower but rises faster than the red one.

In [ ]:
# =============================================================================
# STEP 2 · Draw the two cost lines and find the crossover Q*
# =============================================================================

# ---- 1. Calculate the crossover volume with the formula above --------------
# Q* = extra fixed cost of product focus ÷ variable cost saved per unit
Q_star = (F_product - F_process) / (v_process - v_product)
cost_at_Q_star = total_cost(F_process, v_process, Q_star)   # at Q* both lines have this height

print(f"Crossover volume Q* = {Q_star:,.0f} units per year")   # ",.0f" = thousands separator, no decimals
print(f"At Q* both strategies cost {cost_at_Q_star:,.0f} €")


# ---- 2. Create the volumes for the x-axis ------------------------------------
# A chart needs many points to draw a smooth line.
# np.linspace(0, 20_000, 201) creates 201 evenly spaced numbers: 0, 100, 200, ..., 20,000.
volumes = np.linspace(0, 20_000, 201)

# Because "volumes" is a whole list, total_cost() calculates the cost for ALL 201 volumes
# in one go (numpy does this for us) and returns a list of 201 costs.
costs_process = total_cost(F_process, v_process, volumes)
costs_product = total_cost(F_product, v_product, volumes)


# ---- 3. Draw the chart --------------------------------------------------------
# plt.subplots() creates an empty chart: "fig" is the whole picture, "ax" the drawing area.
fig, ax = plt.subplots(figsize=(9, 5))          # 9 × 5 inches

# ax.plot(x-values, y-values, ...) draws one line.
# We divide the costs by 1,000 so the y-axis shows thousand € (easier to read).
# lw = line width; label = the name shown in the legend.
ax.plot(volumes, costs_process / 1000, color="tab:blue", lw=2.5, label="Process focus")
ax.plot(volumes, costs_product / 1000, color="tab:red",  lw=2.5, label="Product focus")

# Mark the crossover: a black dot (scatter), a dotted vertical line (axvline) and a label (annotate)
ax.scatter([Q_star], [cost_at_Q_star / 1000], color="black", zorder=5)   # zorder=5: draw the dot on top
ax.axvline(Q_star, color="grey", ls=":")                                  # ls=":" = dotted line
ax.annotate(f"Q* = {Q_star:,.0f} units",
            xy=(Q_star, cost_at_Q_star / 1000),                 # the point the text belongs to
            xytext=(Q_star + 800, cost_at_Q_star / 1000 - 150)) # where the text is written

# Axis labels, title, legend and a light grid make the chart readable
ax.set_xlabel("Annual volume Q (units)")
ax.set_ylabel("Total annual cost (thousand €)")
ax.set_title("Process focus vs. product focus")
ax.set_ylim(bottom=0)            # the y-axis starts at 0, so the lines are not exaggerated
ax.legend()
ax.grid(alpha=0.3)               # alpha = transparency (0 invisible, 1 solid)
plt.show()                       # show the finished chart under the cell


**Check your understanding**

1. Why does the blue line start lower but rise faster?
2. A hospital treats about 3,000 patients per year with a given procedure. Which strategy does the chart recommend, and why?
3. Change `F_product` in Step 1 to 700,000, run Step 1 **and** Step 2 again. In which direction does Q\* move? Explain in one sentence.

Re-running two cells for every change is tedious. Step 3 fixes that.


## Step 2b – Aylin's bakery (Backhaus Kaya case, question 5)

Now the same model with real-sized numbers from the case. Aylin compares her **current process** (no extra fixed cost, 0.18 € per roll) with a **semi-automatic roll line** (36,000 € per year for lease and maintenance, 0.10 € per roll). The FrischMarkt contract is 1,800 rolls × 6 days × 50 weeks = **540,000 rolls per year**.

The interactive explorer in Step 3 accepts whole numbers only, so cents like 0.18 € are entered here, in code. **Your turn:** change a number and run the cell again. Which number has to change, and by how much, before the roll line stops paying off for the contract?

In [ ]:
# =============================================================================
# STEP 2b · Aylin's bakery: current process vs. semi-automatic roll line
# =============================================================================
F_current, v_current = 0,      0.18    # current process: no extra fixed cost, 0.18 € per roll
F_line,    v_line    = 36_000, 0.10    # roll line: 36,000 € per year, 0.10 € per roll
contract = 1_800 * 6 * 50               # rolls per year in the FrischMarkt contract = 540,000

# Crossover with the Step 2 formula: extra fixed cost ÷ variable cost saved per roll
Q_bakery = (F_line - F_current) / (v_current - v_line)
print(f"Crossover: {Q_bakery:,.0f} rolls per year")
print(f"Contract:  {contract:,.0f} rolls per year")
print(f"Cost of the contract volume:  current process {total_cost(F_current, v_current, contract):,.0f} €   "
      f"roll line {total_cost(F_line, v_line, contract):,.0f} €")

# The chart: the x-axis must now reach far beyond 20,000, up to 700,000 rolls
rolls = np.linspace(0, 700_000, 141)                      # 0, 5,000, 10,000, ..., 700,000
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rolls, total_cost(F_current, v_current, rolls) / 1000, color="tab:blue", lw=2.5, label="Current process")
ax.plot(rolls, total_cost(F_line, v_line, rolls) / 1000, color="tab:red", lw=2.5, label="Semi-automatic roll line")
ax.axvline(Q_bakery, color="grey", ls=":")                # dotted line at the crossover
ax.axvline(contract, color="black", ls="--", lw=1)        # dashed line at the contract volume
ax.text(Q_bakery, 5, f" crossover {Q_bakery:,.0f}", rotation=90, va="bottom", fontsize=9)
ax.text(contract, 5, f" contract {contract:,.0f}", rotation=90, va="bottom", fontsize=9)
ax.set_xlabel("Rolls per year")
ax.set_ylabel("Total annual cost (thousand €)")
ax.set_title("Backhaus Kaya: which process is cheaper at the contract volume?")
ax.set_ylim(bottom=0)
ax.legend()
ax.grid(alpha=0.3)
plt.show()


## Step 3 – Make the chart interactive

Now we add **widgets**: input elements, like the knobs and dials on a control panel. For every cost figure you get a box where you can type a number, plus a **−** and a **+** button that change the value step by step. Every change redraws the chart immediately.

The next code cell only **defines** three helper functions; it draws nothing yet. You do not have to understand every line. The comments explain what each part does, and the cell after it uses the helpers.

- `cost_control(...)` builds one row: **[ − ] [ number box ] [ + ]**
- `draw_cost_chart(...)` draws all cost lines, shades the cheapest strategy for each volume, and marks where the cheapest strategy changes
- `cost_explorer(...)` puts everything together: input rows for each strategy, plus the chart, connected so that every change triggers a redraw


**What the next cell does, in plain words.**

1. `cost_control` builds one row of the control panel: a minus button, a number box, a plus button. A click changes the number by one step.
2. `draw_cost_chart` takes the current numbers and (a) calculates every strategy's cost at every volume, (b) finds the cheapest strategy at each volume, (c) colours the background accordingly and draws the lines, (d) prints which strategy wins in which volume band and which crossovers matter.
3. `cost_explorer` builds one panel per strategy and **wires** each number box to the chart: when a number changes, the chart is redrawn.

**What you should see:** only the line *"Helpers defined."* The chart appears in the next cell.

In [ ]:
# =============================================================================
# STEP 3 · Three helper functions for the interactive explorer
# =============================================================================
# This cell only DEFINES the helpers; nothing is drawn yet. Think of it as building
# the control panel in the workshop. The cell after it switches the panel on.
#
#   Helper 1  cost_control()     one input row:   [ − ] [ number box ] [ + ]
#   Helper 2  draw_cost_chart()  draws all lines, colours the cheapest strategy, writes a summary
#   Helper 3  cost_explorer()    puts input rows and chart together and connects them


# =============================================================================
# Helper 1: one input row  [ − ] [ number ] [ + ]
# =============================================================================
def cost_control(label, start_value, step):
    """Returns (number_box, row). The row is what we display;
    the number box is what we read the current value from."""

    # A number box for whole numbers between 0 and 10 million.
    # description = the label in front of the box; step = the change per click.
    # (Whole numbers only: for costs like 0.18 € use the Step 2 cells instead.)
    number_box = widgets.BoundedIntText(
        value=start_value, min=0, max=10_000_000, step=step,
        description=label,
        style={"description_width": "130px"},     # width reserved for the label
        layout=widgets.Layout(width="260px"),     # width of label + box together
    )
    minus_button = widgets.Button(description="−", layout=widgets.Layout(width="40px"))
    plus_button  = widgets.Button(description="+", layout=widgets.Layout(width="40px"))

    # What happens on a click: lower or raise the value by one "step".
    # max(0, ...) makes sure the value never drops below zero.
    # (The parameter "button" is required by ipywidgets but not used here.)
    def decrease(button):
        number_box.value = max(0, number_box.value - step)

    def increase(button):
        number_box.value = number_box.value + step

    # on_click connects a button to the function that should run when it is clicked
    minus_button.on_click(decrease)
    plus_button.on_click(increase)

    row = widgets.HBox([minus_button, number_box, plus_button])   # HBox = place side by side
    return number_box, row


# =============================================================================
# Helper 2: draw the chart for any number of strategies
# =============================================================================
def draw_cost_chart(strategies, values, max_volume, y_max):
    """strategies: dictionary with names and colours
       values:     dictionary  name -> (F, v)  with the CURRENT numbers
       max_volume: right end of the x-axis
       y_max:      top of the y-axis in €"""

    names = list(strategies)                        # the strategy names, e.g. ["Process focus", "Product focus"]
    volumes = np.arange(0, max_volume + 1, 10)      # 0, 10, 20, ..., max_volume (one point every 10 units)

    # --- a) total cost of every strategy at every volume -------------------
    # The square brackets build a list: one cost line per strategy.
    # np.array turns it into a table: one ROW per strategy, one COLUMN per volume.
    costs = np.array([total_cost(values[n][0], values[n][1], volumes) for n in names])

    # --- b) for every volume: which strategy is cheapest? -------------------
    # argmin looks down each column and returns the row number of the smallest cost,
    # e.g. [0, 0, 0, ..., 1, 1] means: strategy 0 is cheapest first, later strategy 1.
    cheapest = costs.argmin(axis=0)

    # Find the volume bands where the cheapest strategy stays the same.
    # We walk along the volumes; whenever the winner changes, one band ends and the next starts.
    # Each band is stored as (name, start volume, end volume).
    bands = []
    band_start = 0
    for i in range(1, len(volumes) + 1):
        if i == len(volumes) or cheapest[i] != cheapest[band_start]:
            bands.append((names[cheapest[band_start]], volumes[band_start], volumes[i - 1]))
            band_start = i

    # --- c) draw ------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 5.5))

    # Light background colour = the strategy that is cheapest in that band
    # (axvspan colours a vertical stripe from "start" to "end"; alpha=0.08 makes it very light)
    for name, start, end in bands:
        ax.axvspan(start, end, color=strategies[name]["color"], alpha=0.08)

    # One line per strategy, in its own colour (dashed if the dictionary asks for it)
    for row, name in enumerate(names):              # enumerate gives the row number AND the name
        style = strategies[name].get("linestyle", "-")   # "-" = solid line if nothing else is given
        ax.plot(volumes, costs[row] / 1000, color=strategies[name]["color"],
                lw=2.5, ls=style, label=name)

    # Exact switch points between neighbouring bands, using the Step 2 formula Q* = ΔF ÷ Δv
    switch_points = []
    for (name_a, _, _), (name_b, _, _) in zip(bands[:-1], bands[1:]):   # each band with the next one
        F_a, v_a = values[name_a]
        F_b, v_b = values[name_b]
        Q_star = (F_b - F_a) / (v_a - v_b)
        switch_points.append((name_a, name_b, Q_star))
        ax.axvline(Q_star, color="grey", ls=":")                                    # dotted line
        ax.scatter([Q_star], [total_cost(F_a, v_a, Q_star) / 1000], color="black", zorder=5)  # dot
        ax.text(Q_star, y_max / 1000 * 0.97, f" {Q_star:,.0f}", va="top", fontsize=9)       # number at the top

    ax.set_xlim(0, max_volume)
    ax.set_ylim(0, y_max / 1000)
    ax.set_xlabel("Annual volume Q (units)")
    ax.set_ylabel("Total annual cost (thousand €)")
    ax.set_title("Background colour = cheapest strategy at that volume")
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)
    plt.show()

    # --- d) a short text summary under the chart -----------------------------
    # Band limits: 0, the exact switch points, and the right end of the axis
    limits = [0] + [q for _, _, q in switch_points] + [max_volume]
    print("Cheapest strategy by volume band:")
    for k, (name, _, _) in enumerate(bands):
        print(f"   {limits[k]:>7,.0f} – {limits[k + 1]:>7,.0f} units:  {name}")
    never = [n for n in names if n not in [b[0] for b in bands]]   # strategies that never win
    for n in never:
        print(f"   {n} is never the cheapest in this volume range.")

    # All pairwise crossovers, and whether they actually matter
    print("\nAll pairwise crossover volumes:")
    for i in range(len(names)):
        for j in range(i + 1, len(names)):          # every pair once: (0,1), (0,2), (1,2), ...
            F_a, v_a = values[names[i]]
            F_b, v_b = values[names[j]]
            pair = f"{names[i]} = {names[j]}"
            if v_a == v_b:                          # same slope: the lines never meet
                print(f"   {pair}: lines are parallel, they never cross")
                continue                            # "continue" = skip to the next pair
            Q_star = (F_b - F_a) / (v_a - v_b)
            if Q_star < 0:                          # they would meet only at a negative volume
                print(f"   {pair}: no crossover at a positive volume")
                continue
            # A crossover matters only if no third strategy is cheaper at that volume
            cost_here = total_cost(F_a, v_a, Q_star)
            cheapest_here = min(total_cost(*values[n], Q_star) for n in names)
            if Q_star > max_volume:
                note = "(beyond the right edge of the chart)"
            elif cost_here <= cheapest_here + 1e-6:  # + 1e-6: tolerate tiny rounding differences
                note = "→ switch point"
            else:
                note = "(irrelevant: another strategy is cheaper there)"
            print(f"   {pair}: at {Q_star:>9,.0f} units  {note}")


# =============================================================================
# Helper 3: put input rows and chart together
# =============================================================================
def cost_explorer(strategies, max_volume=40_000, y_max=1_600_000,
                  fixed_step=25_000, variable_step=2):
    """Builds the interactive explorer for the strategies given.
    The values after "=" are defaults: used unless you give other numbers."""

    boxes = {}          # name -> (fixed-cost box, variable-cost box)
    panels = []         # one small panel of inputs per strategy

    # For every strategy: two input rows (fixed cost, variable cost) and a coloured heading
    for name, s in strategies.items():
        F_box, F_row = cost_control("Fixed cost (€/yr)", s["F"], fixed_step)
        v_box, v_row = cost_control("Variable (€/unit)", s["v"], variable_step)
        boxes[name] = (F_box, v_box)
        heading = widgets.HTML(f"<b style='color:{s['color']}'>{name}</b>")
        panels.append(widgets.VBox([heading, F_row, v_row]))    # VBox = stack on top of each other

    chart_area = widgets.Output()     # a placeholder where the chart appears

    # Read all current numbers, then redraw the chart inside the placeholder
    def redraw(change=None):
        values = {n: (boxes[n][0].value, boxes[n][1].value) for n in strategies}
        with chart_area:
            clear_output(wait=True)   # remove the old chart first
            draw_cost_chart(strategies, values, max_volume, y_max)

    # "observe": call redraw() whenever a number box changes its value.
    # This is the wire between the control panel and the chart.
    for F_box, v_box in boxes.values():
        F_box.observe(redraw, names="value")
        v_box.observe(redraw, names="value")

    # Show input panels in rows of two, then the chart below them
    rows = [widgets.HBox(panels[i:i + 2]) for i in range(0, len(panels), 2)]
    display(widgets.VBox(rows), chart_area)
    redraw()                          # draw once at the start


print("Helpers defined. Run the next cell to start the explorer.")


### 3b – Explore the two extremes

Run the cell below. Use the **−** / **+** buttons (fixed cost changes by 25,000 €, variable cost by 2 € per click) or type a number and press Enter.


**What the next cell does.** It writes the starting values of the two strategies into a *dictionary* (a small table with names) and switches on the explorer. **What you should see:** two input panels and the chart from Step 2, now with a coloured background: blue where process focus is cheaper, red where product focus is cheaper. The text under the chart lists the volume bands.

In [ ]:
# A "dictionary" stores information under names, like a small address book:
# the name in quotes is the key, and after the colon come the values for that key.
# For each strategy: F = fixed cost [€/year], v = variable cost [€/unit], and a colour.
two_strategies = {
    "Process focus": {"F":  50_000, "v": 60, "color": "tab:blue"},
    "Product focus": {"F": 500_000, "v": 10, "color": "tab:red"},
}

# Switch on the control panel from Step 3 with these two strategies.
# Change the numbers with − / + or type a new whole number and press Enter.
cost_explorer(two_strategies)


**Experiments**

1. Click **+** on the fixed cost of product focus four times. Where does the crossover move, and why?
2. Reset (re-run the cell), then lower the variable cost of product focus to 4 €. What happens, and what real-world investment could cause such a drop?
3. Find a setting where product focus is cheaper at **every** volume shown. Is that realistic?
4. *Discussion:* An automation vendor promises to cut your variable cost by half. What question should you ask before you believe that it pays off?


## Step 4 – Add the middle: repetitive focus

Between the two extremes sits **repetitive focus** (e.g. Harley-Davidson, car assembly, fast food): standard modules assembled on a line. Its fixed cost and its variable cost both lie **between** the two extremes.

Nothing in the code changes except the dictionary: we add one more entry. The helpers from Step 3 work for any number of strategies.


**What the next cell does.** It adds one line to the dictionary and calls the same explorer. **What you should see:** three coloured background bands (blue, green, red) and a summary that lists three pairwise crossovers, of which only two are switch points.

In [ ]:
# The same dictionary as in 3b, with ONE new entry in the middle.
# Nothing else changes: the helpers from Step 3 work for any number of strategies.
three_strategies = {
    "Process focus":    {"F":  50_000, "v": 60, "color": "tab:blue"},
    "Repetitive focus": {"F": 200_000, "v": 25, "color": "tab:green"},   # NEW: between the two extremes
    "Product focus":    {"F": 500_000, "v": 10, "color": "tab:red"},
}

cost_explorer(three_strategies)   # read the summary under the chart: which crossovers are switch points?


**Experiments**

1. Read the text under the chart. Three lines create three pairwise crossovers, but only **two** are switch points. Which one is irrelevant, and why?
2. Raise the fixed cost of repetitive focus in steps of 25,000 €. At what value does the green band disappear completely? What would that mean for a company like Harley-Davidson?
3. Each strategy now "owns" a volume band. Write a one-sentence decision rule a manager could use.


## Step 5 – Add mass customization

**Mass customization** (e.g. Dell building each computer to order) aims for high variety **and** high volume. It needs heavy up-front investment in modular product design, flexible automation, IT and a responsive supply chain, so its **fixed cost is the highest of all**. Its variable cost per unit is low, close to product focus, but usually a little higher, because each unit is configured individually.

The dashed purple line shows it.


**What the next cell does.** Same as before, with a fourth entry. The only new thing is `"linestyle": "--"`: a dashed line for mass customization. Watch the summary under the chart: does mass customization own any volume band?

In [ ]:
# Four strategies. "linestyle": "--" draws mass customization as a dashed line,
# so you can tell it apart from product focus, whose cost line runs close to it.
four_strategies = {
    "Process focus":      {"F":  50_000, "v": 60, "color": "tab:blue"},
    "Repetitive focus":   {"F": 200_000, "v": 25, "color": "tab:green"},
    "Product focus":      {"F": 500_000, "v": 10, "color": "tab:red"},
    "Mass customization": {"F": 800_000, "v": 12, "color": "tab:purple",
                           "linestyle": "--"},                          # NEW: highest fixed cost of all
}

cost_explorer(four_strategies)


**Experiments and the big question**

1. Look at the summary under the chart. In which volume band is mass customization the cheapest strategy?
2. Lower the variable cost of mass customization to 6 € and read the line *Product focus = Mass customization* in the summary. At what volume would mass customization finally beat product focus? Why does it need such a high volume?
3. **The big question:** with the starting values, the chart says nobody should ever choose mass customization. Yet Dell built a highly successful business on it. What does the cost chart leave out?

<details>
<summary><b>Hint (click to open)</b></summary>

The chart has only one axis along the bottom: **volume**. It compares **cost** only. Mass customization does not win on cost; it wins on **variety**, the ability to sell thousands of configurations to customers who want something tailored, often at a higher price. That benefit shows up as **revenue**, and the cost chart has no axis for it. A model is useful because it simplifies; the skill lies in knowing **what** it leaves out.
</details>


## Summary

- Every process strategy has a cost line: **Total cost = F + v × Q**.
- Process focus: low F, high v. Product focus: high F, low v. Repetitive focus: in between.
- Lines cross at **Q\* = ΔF ÷ Δv**; each strategy owns the volume band where its line is lowest.
- Not every crossover matters: a third strategy can be cheaper at that point.
- Mass customization looks unattractive in a pure cost model. Its advantage, variety and the revenue it brings, lies on an axis the model does not have.

**Source:** Heizer, J., Render, B., & Munson, C. (2023). *Operations management: Sustainability and supply chain management* (14th ed.). Pearson. https://www.pearson.com/en-us/subject-catalog/p/operations-management-sustainability-and-supply-chain-management/P200000007031/9780137649556 – Chapter 7 (Process Strategies). All cost figures in this notebook are illustrative.

---
*Transparency note: Claude Opus 5.5 was used to draft the code, comments and exercises of this notebook and to check it for errors. All contents were reviewed, revised and are the responsibility of Roman Bartnik.*
